In [1]:
# 텍스트 추출 
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

pdf_lg_aimers = "data/LG Aimers 4기 소개자료.pdf"
txt_loader = PyMuPDF4LLMLoader(
    file_path=pdf_lg_aimers,
    mode="page"  # 페이지별로 분리
)


In [2]:
txt_docs = txt_loader.load()
print(f"PDF 파일의 페이지 수: {len(txt_docs)}")


Consider using the pymupdf_layout package for a greatly improved page layout analysis.
PDF 파일의 페이지 수: 7


In [3]:
txt_docs[1].metadata


{'producer': '',
 'creator': '',
 'creationdate': '2025-07-15T07:40:32+00:00',
 'source': 'data/LG Aimers 4기 소개자료.pdf',
 'file_path': 'data/LG Aimers 4기 소개자료.pdf',
 'total_pages': 7,
 'format': 'PDF 1.7',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2025-07-15T07:40:32+00:00',
 'trapped': '',
 'modDate': "D:20250715074032+00'00'",
 'creationDate': "D:20250715074032+00'00'",
 'page': 1}

In [4]:
print(txt_docs[1].page_content)

#### **1. LG Aimers 프로그램개요**

###### ❑ 교육과 경험의 기회를 필요로 하는 청년들에게 양질의 AI교육을 온라인으로 제공하고, 기업의 실제 data를 다루며 실무를 경험할 수 있는 기회를 제공하기 위해 『온라인 AI전문가 과정(1개월)』과『AI 해커톤 (Hackathon) (1개월)』을 **결합한 형태의 교육 프로그램** **-** 온라인 AI 교육과정: 국내 AI 전문가 (현업 전문가, 저명 교수) 의 최신 AI 기법에관한 온라인 강의(1개월)


     **온라인 초급 교육은 기초적인 프로그래밍데이터 처리에 관련된 내용들로써 외부 교육 콘텐츠를 활용(optional)**


          **초급 교육은 자료구조와 알고리즘 등 컴퓨팅 관련 기본 소양 교육**


     **온라인 중상급 교육은 국내 AI 분야전문가들이 주제별로 강의 (LG 자체 개발-** **부록 참조)**


          **중상급 교육은 대학원 수준의 AI 요소 기술 및 이론에 관한 교육으로 이루어져 있음**


          **중상급 교육을 받기 전에 학부 수준의 ‘인공지능 개론’ 과목을 이수하는 것을 추천**

###### **-** **온라인 AI 해커톤: LG계열사의문제를 현장의 실제data를활용하여 해결하는 해커톤 예선(1개월)** **-** **오프라인 AI 해커톤: 온라인 해커톤에서 선발된 본선 진출자들만 참가 (1박2일)** ❑ 수료 조건 : 온라인 AI전문가 과정이수(교육영상100% 이수및퀴즈참여) & AI 해커톤 baseline model 성능이상 **수료자에 한하여 수료증 발급** ❑ 대상 및 규모 : 만 19세에서 29세의 청년 (미취업자 대상) ❑ 운영 일정 : 매년 연 2회 진행. 여름방학(7~8월), 겨울방학(1~2월)


❑ **LG Aimers 채널:** **[https://lgaimers.ai/](https://lgaimers.ai/)**





# 이미지 설명 텍스트 추가 
> PDF -> 이미지 추출 -> GPT 모델이 이미지 분석 -> 이미지 설명 텍스트 생성 -> page_content에 포함 

In [5]:
from dotenv import load_dotenv

load_dotenv()

False

In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-mini",   # 사용할 OpenAI 챗 모델
    max_tokens=1024       # 생성할 응답의 최대 토큰 수
)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
from langchain_community.document_loaders.parsers import LLMImageBlobParser


img_loader = PyMuPDF4LLMLoader(
    pdf_lg_aimers,        # 로드할 PDF 파일 경로 또는 바이너리
    mode="page",          # 페이지 단위로 Document를 생성하도록 설정
    extract_images=True,  # PDF 내 포함된 이미지를 추출하도록 설정
    images_parser=LLMImageBlobParser(
        model=llm         # 추출된 이미지를 LLM 기반 파서로 분석
                            # (예: 이미지 캡션 생성, 이미지 내용 설명 등 가능)
    ),
)

In [ ]:
img_docs = img_loader.load()
print(f"PDF 파일의 페이지 수: {len(img_docs)}")

NameError: name 'img_loader' is not defined

In [ ]:
print(img_docs[2].page_content[:300])

In [ ]:
import fitz  # PyMuPDF
from pathlib import Path

# 이미지 저장 폴더 생성
output_dir = Path("extracted_images")
output_dir.mkdir(exist_ok=True)

# PDF 문서 열기
pdf_doc = fitz.open(pdf_lg_aimers)

# 각 페이지별로 이미지 추출 및 메타데이터 추가
for page_num in range(len(pdf_doc)):
    page = pdf_doc[page_num]
    image_list = page.get_images()
    
    # 해당 페이지의 이미지 파일명 리스트
    extracted_image_files = []
    
    # 페이지의 각 이미지 추출
    for img_index, img in enumerate(image_list):
        xref = img[0]  # 이미지 참조 번호
        
        try:
            # 이미지 추출
            base_image = pdf_doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]  # 이미지 확장자 (png, jpg 등)
            
            # 이미지 파일명 생성
            image_filename = f"page_{page_num}_img_{img_index}.{image_ext}"
            image_path = output_dir / image_filename
            
            # 이미지 저장
            with open(image_path, "wb") as img_file:
                img_file.write(image_bytes)
            
            extracted_image_files.append(str(image_path))
            
        except Exception as e:
            print(f"페이지 {page_num}, 이미지 {img_index} 추출 실패: {e}")
    
    # 해당 페이지의 Document 메타데이터에 이미지 파일명 추가
    if page_num < len(img_docs):
        img_docs[page_num].metadata['extracted_images'] = extracted_image_files
        img_docs[page_num].metadata['image_count'] = len(extracted_image_files)

pdf_doc.close()

print(f"이미지 추출 완료!")
print(f"저장 위치: {output_dir.absolute()}")
